# Workshop Notebook 2: Model Diagnostics and Architecture Augmentation

**CIROH Developer's Conference 2026 | Foundations of Machine Learning**

---

## Overview

A model that achieves NSE > 0.7 on average is promising — but the *mean* hides a lot. In this notebook we will:

1. **Diagnose** — where and when does the baseline LSTM fail?
2. **Augment with static attributes** — give the model information about basin characteristics
3. **Deepen the architecture** — add layers and explore structural modifications
4. **Compare** — side-by-side evaluation of all variants

## When should you modify a model's architecture?

Signs that structural augmentation may help:
- Systematic bias (model always over- or under-predicts in certain conditions)
- Consistent failure across seasons or flow regimes (e.g., always misses peaks)
- Large inter-basin NSE variance that correlates with a basin attribute

Signs that architecture is *not* the bottleneck:
- Training loss is much lower than validation loss -> regularization problem first
- Some basins have very few observations -> data quality problem


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cpu")

ROOT = Path("..").resolve()
DATA_DIR = ROOT / "data"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(Path(".")))

from camels_loader import CamelsSubsetLoader, FORCING_NAMES, ATTRIBUTE_NAMES
from utils import (
    StreamflowDataset,
    masked_mse_loss,
    nse_score,
    count_params,
    train_model,
    train_model_with_attrs,
    predict_full_timeseries,
    predict_ts_with_attrs,
)

print("Imports OK.")


In [ ]:
# Reproduce the preprocessing from Notebook 1

loader = CamelsSubsetLoader(
    pickle_path=str(DATA_DIR / "camels_daymetv2"),
    gage_id_path=str(DATA_DIR / "gage_id.npy"),
)

TRAIN_END = "1999-09-30"
VAL_END = "2008-09-30"
SEQ_LEN = 365
STRIDE = 7

dates_idx = pd.DatetimeIndex(loader.dates)
train_mask = dates_idx <= pd.Timestamp(TRAIN_END)
val_mask = (dates_idx > pd.Timestamp(TRAIN_END)) & (dates_idx <= pd.Timestamp(VAL_END))
test_mask = dates_idx > pd.Timestamp(VAL_END)
dates_test = loader.dates[test_mask]

x_train = loader.forcings[train_mask]
x_val = loader.forcings[val_mask]
x_test = loader.forcings[test_mask]
y_train = loader.target[train_mask]
y_val = loader.target[val_mask]
y_test = loader.target[test_mask]

x_mean = x_train.mean(axis=(0, 1), keepdims=True)
x_std = x_train.std(axis=(0, 1), keepdims=True) + 1e-8
x_train_norm = (x_train - x_mean) / x_std
x_val_norm = (x_val - x_mean) / x_std
x_test_norm = (x_test - x_mean) / x_std

y_log_mean = float(np.nanmean(np.log1p(np.clip(y_train, 0, None))))
y_log_std = float(np.nanstd(np.log1p(np.clip(y_train, 0, None))[~np.isnan(y_train)])) + 1e-8

def normalize_target(y):
    return (np.log1p(np.clip(y, 0, None)) - y_log_mean) / y_log_std

def denormalize_target(y_norm):
    return np.expm1(y_norm * y_log_std + y_log_mean)

y_train_norm = normalize_target(y_train)
y_val_norm = normalize_target(y_val)
y_test_norm = normalize_target(y_test)
obs_cfs_test = loader.target[test_mask, :, 0]
N_FEATURES = len(FORCING_NAMES)

train_loader = DataLoader(
    StreamflowDataset(x_train_norm, y_train_norm, SEQ_LEN, STRIDE), 128, shuffle=True, drop_last=True)
val_loader = DataLoader(
    StreamflowDataset(x_val_norm, y_val_norm, SEQ_LEN, STRIDE), 128)

print(loader)


In [ ]:
# Train the baseline model to use as a reference point

class LstmModel(nn.Module):
    def __init__(self, n_features, hidden_size=64, n_layers=1, dropout=0.0):
        super().__init__()
        self.input_proj = nn.Linear(n_features, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, n_layers,
                            dropout=dropout if n_layers > 1 else 0.0,
                            batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.output_proj = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = torch.relu(self.input_proj(x))
        out, _ = self.lstm(x)
        return self.output_proj(self.dropout(out)).squeeze(-1)


baseline = LstmModel(N_FEATURES, hidden_size=64, dropout=0.4).to(device)
print(f"Baseline parameters: {count_params(baseline):,}")
print("Training baseline...\n")

tl_base, vl_base = train_model(baseline, train_loader, val_loader, n_epochs=30)

pred_base_cfs = denormalize_target(predict_full_timeseries(baseline, x_test_norm, SEQ_LEN))
nse_base = {gid: nse_score(pred_base_cfs[:, i], obs_cfs_test[:, i])
            for i, gid in enumerate(loader.gage_ids)}

print("\nBaseline NSE (test):")
for gid, nse in nse_base.items():
    print(f"  {gid}: {nse:.3f}")
print(f"  Mean: {np.nanmean(list(nse_base.values())):.3f}")



---
## 1. Model Diagnostics

Before making any architectural changes, we should understand *where* and *why* the model struggles. Good diagnostics prevent us from making changes that don't address the actual problem.

We'll examine:
- **Residual patterns** — does the error have structure (seasonal, flow-magnitude)?
- **Flow duration curves** — does the model capture the full distribution of flow?
- **Basin-level correlation** — do NSE scores correlate with basin attributes?


In [ ]:
# Seasonal bias: residuals by calendar month
errors = pred_base_cfs - obs_cfs_test  # positive = over-prediction
months = pd.DatetimeIndex(dates_test).month

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

monthly_errors = [errors[months == m, :].ravel() for m in range(1, 13)]
monthly_errors = [e[~np.isnan(e)] for e in monthly_errors]

axes[0].boxplot(monthly_errors, labels=list("JFMAMJJASOND"),
                showfliers=False, patch_artist=True,
                boxprops=dict(facecolor="steelblue", alpha=0.6))
axes[0].axhline(0, color="red", linewidth=1.5, linestyle="--")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Prediction error (ft^3/s)")
axes[0].set_title("Seasonal Bias")
axes[0].grid(alpha=0.3)

obs_flat = obs_cfs_test.ravel()
err_flat = errors.ravel()
mask = ~np.isnan(obs_flat) & ~np.isnan(err_flat)
axes[1].hexbin(np.log1p(obs_flat[mask]), err_flat[mask],
               gridsize=40, cmap="Blues", mincnt=1)
axes[1].axhline(0, color="red", linewidth=1.5, linestyle="--")
axes[1].set_xlabel("log(1 + Observed streamflow)")
axes[1].set_ylabel("Prediction error (ft^3/s)")
axes[1].set_title("Error vs. Flow Magnitude")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Flow duration curves — exceedance probability
# A good model should match the full distribution, not just the mean.

fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for i, (ax, gid) in enumerate(zip(axes.ravel(), loader.gage_ids)):
    obs = obs_cfs_test[:, i]
    pred = pred_base_cfs[:, i]
    mask = ~np.isnan(obs) & ~np.isnan(pred)

    obs_sorted = np.sort(obs[mask])[::-1]
    pred_sorted = np.sort(pred[mask])[::-1]
    ep = np.linspace(0, 1, len(obs_sorted))

    ax.semilogy(ep, obs_sorted, color="navy", lw=1.2, label="Observed")
    ax.semilogy(ep, pred_sorted, color="tomato", lw=1.2, linestyle="--", label="Predicted")
    ax.set_title(f"{gid}\nNSE={nse_base[gid]:.2f}", fontsize=9)
    ax.grid(alpha=0.2, which="both")
    if i == 0:
        ax.legend(fontsize=7)
    if i >= 5:
        ax.set_xlabel("Exceedance prob.", fontsize=8)
    if i % 5 == 0:
        ax.set_ylabel("Streamflow (ft^3/s)", fontsize=8)

fig.suptitle("Flow Duration Curves — Baseline", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Does NSE correlate with basin attributes?
# This helps identify what information the model is missing.

nse_arr = np.array(list(nse_base.values()))
attr_keys = ["area_gages2", "frac_forest", "aridity", "elev_mean", "soil_porosity", "frac_snow"]

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, key in zip(axes.ravel(), attr_keys):
    attr_vals = loader.attributes[:, ATTRIBUTE_NAMES.index(key)]
    r, p = stats.pearsonr(attr_vals, nse_arr)
    ax.scatter(attr_vals, nse_arr, color="steelblue", s=60, zorder=3)
    for j, gid in enumerate(loader.gage_ids):
        ax.annotate(str(gid)[-4:], (attr_vals[j], nse_arr[j]),
                    fontsize=7, ha="center", va="bottom")
    ax.set_xlabel(key, fontsize=9)
    ax.set_ylabel("NSE", fontsize=9)
    ax.set_title(f"r = {r:.2f}  (p={p:.2f})", fontsize=9)
    ax.axhline(0, color="gray", lw=0.8, linestyle="--")
    ax.grid(alpha=0.3)

fig.suptitle("NSE vs. Basin Attributes — Where Does the Baseline Struggle?",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


**Discussion questions:**
- Is there a seasonal pattern in the errors? What physical process might cause this?
- Does the model over- or under-predict high-flow events (left end of the FDC)?
- Which basin attribute most strongly correlates with poor NSE? What might be missing?

One common finding: models trained without **static basin attributes** struggle most on atypical basins (high aridity, high snow fraction, large area). This motivates the next section.



---
## 2. Augmentation: Static Basin Attributes

The baseline model only sees **dynamic** climate forcings — it has no knowledge of
basin physical characteristics. Two basins receiving identical precipitation can respond
very differently depending on soil permeability, slope, and vegetation.

By embedding static attributes into the model, we allow it to *condition* its
predictions on those characteristics.

### Architecture modification

```
Baseline:
  x_dynamic  (batch, seq_len, 6) -> LSTM -> prediction

With attributes:
  x_dynamic (batch, seq_len, 6)                                
  x_static (batch, 35) -> MLP -> repeat across time | cat -> LSTM  -> prediction
```


In [ ]:
# Normalize static attributes
attrs_raw = loader.attributes.astype(np.float32) # (10, 35)
attrs_mean = attrs_raw.mean(axis=0, keepdims=True)
attrs_std = attrs_raw.std(axis=0, keepdims=True) + 1e-8
attrs_norm = (attrs_raw - attrs_mean) / attrs_std

N_ATTRS = attrs_norm.shape[1]
print(f"Static attributes: {N_ATTRS} features per basin")


class StreamflowDatasetWithAttrs(Dataset):
    """Extends StreamflowDataset to also return per-basin static attributes.
    Each sample: (x_dynamic, x_static, y)
    """
    def __init__(self, x, y, attrs, seq_len=365, stride=1):
        self.samples = []
        n_time, n_basins, _ = x.shape
        for basin in range(n_basins):
            for t in range(0, n_time - seq_len, stride):
                self.samples.append((
                    x[t:t + seq_len, basin, :].astype(np.float32),
                    attrs[basin].astype(np.float32),
                    y[t:t + seq_len, basin, 0].astype(np.float32),
                ))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        x_dyn, x_stat, y = self.samples[idx]
        return torch.from_numpy(x_dyn), torch.from_numpy(x_stat), torch.from_numpy(y)


train_ds_a = StreamflowDatasetWithAttrs(x_train_norm, y_train_norm, attrs_norm, SEQ_LEN, STRIDE)
val_ds_a = StreamflowDatasetWithAttrs(x_val_norm, y_val_norm, attrs_norm, SEQ_LEN, STRIDE)
train_loader_a = DataLoader(train_ds_a, batch_size=128, shuffle=True, drop_last=True)
val_loader_a = DataLoader(val_ds_a, batch_size=128)

x_dyn, x_stat, y_b = next(iter(train_loader_a))
print(f"x_dynamic: {tuple(x_dyn.shape)} (batch, seq_len, forcings)")
print(f"x_static: {tuple(x_stat.shape)} (batch, n_attrs)")
print(f"y: {tuple(y_b.shape)} (batch, seq_len)")


In [ ]:
class LstmWithAttrs(nn.Module):
    """LSTM conditioned on static basin attributes.

    The attribute vector is encoded into an embedding and concatenated with the
    dynamic forcings at every timestep, enabling basin-specific behaviour.

    Parameters
    ----------
    n_features : number of dynamic forcing variables
    n_attrs : number of static basin attributes
    hidden_size : LSTM hidden units
    attr_embed : dimension to project attributes to before concatenation
    dropout : dropout probability
    """

    def __init__(self, n_features: int, n_attrs: int,
                 hidden_size: int = 64, attr_embed: int = 32,
                 dropout: float = 0.0):
        super().__init__()
        self.attr_encoder = nn.Sequential(
            nn.Linear(n_attrs, attr_embed),
            nn.Tanh(),
        )
        self.input_proj = nn.Linear(n_features + attr_embed, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.output_proj = nn.Linear(hidden_size, 1)

    def forward(self, x_dyn: torch.Tensor, x_stat: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        x_dyn : (batch, seq_len, n_features)
        x_stat : (batch, n_attrs)

        Returns
        -------
        (batch, seq_len)
        """
        attr_emb = self.attr_encoder(x_stat)
        attr_emb = attr_emb.unsqueeze(1).expand(-1, x_dyn.size(1), -1)
        x = torch.relu(self.input_proj(torch.cat([x_dyn, attr_emb], dim=-1)))
        out, _ = self.lstm(x)
        return self.output_proj(self.dropout(out)).squeeze(-1)


model_attrs = LstmWithAttrs(
    n_features=N_FEATURES, n_attrs=N_ATTRS,
    hidden_size=64, attr_embed=32, dropout=0.4
).to(device)

print(f"LstmWithAttrs parameters: {count_params(model_attrs):,}")
print("Training...\n")

tl_attrs, vl_attrs = train_model_with_attrs(
    model_attrs, train_loader_a, val_loader_a, n_epochs=30
)

pred_attrs_cfs = denormalize_target(
    predict_ts_with_attrs(model_attrs, x_test_norm, attrs_norm, SEQ_LEN)
)
nse_attrs = {gid: nse_score(pred_attrs_cfs[:, i], obs_cfs_test[:, i])
             for i, gid in enumerate(loader.gage_ids)}

print("\nNSE — Baseline vs. +Attributes:")
print(f"  {'Gage ID':>10s}  {'Baseline':>10s}  {'+Attrs':>10s}  {'Δ NSE':>8s}")
for gid in loader.gage_ids:
    delta = nse_attrs[gid] - nse_base[gid]
    print(f"  {gid:>10d}  {nse_base[gid]:10.3f}  {nse_attrs[gid]:10.3f}  {delta:+8.3f}")
print(f"  {'Mean':>10s}  {np.nanmean(list(nse_base.values())):10.3f}"
      f"  {np.nanmean(list(nse_attrs.values())):10.3f}"
      f"  {np.nanmean(list(nse_attrs.values())) - np.nanmean(list(nse_base.values())):+8.3f}")



---
## 3. Architecture Modifications: Depth and Convolutional Encoders

Two more structural levers:

### Deeper LSTM

Stacking multiple LSTM layers allows each layer to learn a different level of temporal abstraction — short-term runoff dynamics in layer 1, seasonal soil moisture in layer 2, and so on. Depth is only beneficial when the task genuinely has hierarchical temporal structure and when there's enough data to train the extra parameters.

### Causal convolution front-end

Replacing the linear encoder with 1D **causal convolutions** lets the model explicitly aggregate local temporal context (e.g., a 7-day precipitation event) before the LSTM processes the sequence. "Causal" means each output only depends on *past* inputs — no future leakage.

```
Input (batch, seq_len, features)
 │
 ▼  ConstantPad1d (left-pad, causal)
 ▼  Conv1d -> ReLU     looks at kernel_size consecutive days
 ▼  Conv1d -> ReLU     stacks another level of local context
 │
 ▼  LSTM -> Linear -> output
```


In [ ]:
class DeepLstmModel(nn.Module):
    """Two-layer stacked LSTM with inter-layer dropout."""

    def __init__(self, n_features: int, hidden_size: int = 64,
                 n_layers: int = 2, dropout: float = 0.3):
        super().__init__()
        self.input_proj = nn.Linear(n_features, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, n_layers,
                            dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.output_proj = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = torch.relu(self.input_proj(x))
        out, _ = self.lstm(x)
        return self.output_proj(self.dropout(out)).squeeze(-1)


class CausalConvLstmModel(nn.Module):
    """Causal convolution encoder followed by a single-layer LSTM."""

    def __init__(self, n_features: int, hidden_size: int = 64,
                 kernel_size: int = 7, dropout: float = 0.3):
        super().__init__()
        self.pad = nn.ConstantPad1d((kernel_size - 1, 0), 0.0)
        self.conv1 = nn.Conv1d(n_features, hidden_size, kernel_size)
        self.pad2 = nn.ConstantPad1d((kernel_size - 1, 0), 0.0)
        self.conv2 = nn.Conv1d(hidden_size, hidden_size, kernel_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.output_proj = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)                      # (batch, channels, seq_len)
        x = torch.relu(self.conv1(self.pad(x)))
        x = torch.relu(self.conv2(self.pad2(x)))
        x = x.permute(0, 2, 1)                      # (batch, seq_len, hidden)
        out, _ = self.lstm(x)
        return self.output_proj(self.dropout(out)).squeeze(-1)


m1 = DeepLstmModel(N_FEATURES, 64, 2, 0.3).to(device)
m2 = CausalConvLstmModel(N_FEATURES, 64, 7, 0.3).to(device)
print(f"DeepLstmModel       (2-layer LSTM) : {count_params(m1):,} params")
print(f"CausalConvLstmModel (conv + LSTM)  : {count_params(m2):,} params")


In [ ]:
print("=== Training DeepLstmModel ===")
model_deep = DeepLstmModel(N_FEATURES, hidden_size=64, n_layers=2, dropout=0.3).to(device)
tl_deep, vl_deep = train_model(model_deep, train_loader, val_loader, n_epochs=30, verbose=False)
pred_deep_cfs = denormalize_target(predict_full_timeseries(model_deep, x_test_norm, SEQ_LEN))
nse_deep = {gid: nse_score(pred_deep_cfs[:, i], obs_cfs_test[:, i])
            for i, gid in enumerate(loader.gage_ids)}
print(f"  Mean NSE: {np.nanmean(list(nse_deep.values())):.3f}\n")

print("=== Training CausalConvLstmModel ===")
model_conv = CausalConvLstmModel(N_FEATURES, hidden_size=64, kernel_size=7, dropout=0.3).to(device)
tl_conv, vl_conv = train_model(model_conv, train_loader, val_loader, n_epochs=30, verbose=False)
pred_conv_cfs = denormalize_target(predict_full_timeseries(model_conv, x_test_norm, SEQ_LEN))
nse_conv = {gid: nse_score(pred_conv_cfs[:, i], obs_cfs_test[:, i])
            for i, gid in enumerate(loader.gage_ids)}
print(f"  Mean NSE: {np.nanmean(list(nse_conv.values())):.3f}")


> **Exercise** — Add your own modification.
>
> Ideas to try:
> - **Larger kernel** — change `kernel_size` in `CausalConvLstmModel` to 14 or 30
> - **Residual connection** — add the input back to the LSTM output: `out = lstm_out + input_proj(x)`
> - **Combine attributes + deep** — modify `LstmWithAttrs` to use 2 LSTM layers
> - **Highway LSTM** — add a gating mechanism on the skip connection around the LSTM



---
## 4. Model Comparison

| Model | Key feature | Expected benefit |
|-------|-------------|-----------------|
| **Baseline** | hidden=64, 1 layer | Reference point |
| **+Attributes** | Static basin embeddings | Better inter-basin generalization |
| **DeepLSTM** | 2 stacked LSTM layers | Hierarchical temporal features |
| **ConvLSTM** | Causal conv encoder + LSTM | Explicit local pattern extraction |


In [ ]:
all_models = {
    "Baseline": nse_base,
    "+Attributes": nse_attrs,
    "DeepLSTM": nse_deep,
    "ConvLSTM": nse_conv,
}

df_nse = pd.DataFrame(all_models, index=loader.gage_ids)
df_nse.index.name = "Gage ID"
df_nse.loc["Mean"] = df_nse.mean()
df_nse.loc["Median"] = df_nse.iloc[:-1].median()

df_nse.round(3).style.background_gradient(cmap="RdYlGn", vmin=0, vmax=1)


In [ ]:
mean_nse = {name: np.nanmean(list(scores.values())) for name, scores in all_models.items()}
colors = ["#7f8c8d", "#2980b9", "#27ae60", "#8e44ad"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

bars = axes[0].bar(mean_nse.keys(), mean_nse.values(), color=colors, alpha=0.85)
axes[0].axhline(0, color="k", lw=0.8)
axes[0].set_ylabel("Mean NSE (test period)")
axes[0].set_title("Mean NSE — All Variants")
axes[0].set_ylim(min(mean_nse.values()) - 0.1, 1.0)
for bar, val in zip(bars, mean_nse.values()):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                 f"{val:.3f}", ha="center", fontsize=10, fontweight="bold")
axes[0].grid(alpha=0.3, axis="y")

for (name, nse_d), c in zip(all_models.items(), colors):
    vals = [nse_d[gid] for gid in loader.gage_ids]
    axes[1].scatter(range(len(loader.gage_ids)), vals, label=name, color=c, s=60, zorder=3)
    axes[1].plot(range(len(loader.gage_ids)), vals, color=c, lw=1, alpha=0.6)

axes[1].set_xticks(range(len(loader.gage_ids)))
axes[1].set_xticklabels([str(g) for g in loader.gage_ids], rotation=45, ha="right", fontsize=8)
axes[1].set_ylabel("NSE")
axes[1].set_title("Per-Basin NSE — All Variants")
axes[1].axhline(0, color="gray", lw=0.8, linestyle="--")
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("model_comparison.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Validation loss curves for all variants
lc_data = [
    ("Baseline", vl_base, "#7f8c8d"),
    ("+Attributes", vl_attrs, "#2980b9"),
    ("DeepLSTM", vl_deep, "#27ae60"),
    ("ConvLSTM", vl_conv, "#8e44ad"),
]

fig, ax = plt.subplots(figsize=(10, 4))
for name, vl, c in lc_data:
    ax.plot(vl, color=c, lw=2, label=name)

ax.set_xlabel("Epoch")
ax.set_ylabel("Validation MSE Loss")
ax.set_title("Validation Loss — All Variants")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()



---
## Workshop Summary

Over these two notebooks you have:

1. **Explored** the CAMELS hydrometeorological dataset
2. **Preprocessed** raw time series — temporal splits, normalization, sequence windowing
3. **Built** an LSTM from scratch with `nn.Module`
4. **Trained** with backpropagation, masked loss, and gradient clipping
5. **Evaluated** with Nash-Sutcliffe Efficiency and flow duration curves
6. **Diagnosed** underfitting, overfitting, and seasonal bias
7. **Augmented** with static attributes, deeper LSTM, and causal convolution encoders


---
### What to explore next

- **More basins** — CAMELS has 671 basins; scale up and see how NSE generalizes
- **More features** — add soil moisture indices or snow water equivalent as forcings
- **Transformer** — replace the LSTM with a multi-head self-attention block
- **Differentiable parameter learning** — combine an LSTM with a physics-based model
- **Uncertainty quantification** — use MC Dropout or ensembles to estimate prediction intervals
